# Round 5 · The session queue ⚔️ the ones that got away

Four rounds in, a handful of phishing-kit victims and most of the low-and-slow victims are still unflagged. Nothing links them to anyone and their joint deviation was too weak for the Round 3 model. But they all do what attackers do in the end: **monetize**. Inside one session, on a browser this account has only just started using, they add an address and order to it.

This round is a **review queue judged on its delta**: the top 100 accounts that Round 4 did not already contain, scored for what *they* add (`--prev`). It is also the Round 1 left-censoring trade again, with different numbers.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../tools"))
from atolab import *
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

df = dedupe(load_events())
prev = pd.read_csv("../suspects_r4.csv")      # or the reference: python tools/checkpoint.py --round 4 --catch-up
df = first_seen(df)                            # is_new_device_id, is_new_asn, ...
df["n_prior"] = df.groupby("account_id", observed=True).cumcount()
df["amount"] = pd.to_numeric(df.d_amount, errors="coerce")
u = df[df.actor == "user"].copy()
MIN_HISTORY = 10          # the Round 1 setting. Keep it for the first pass, then change it.
YOUNG_DAYS = 14

## 5.1 Sessions, not events

An attacker's order is one event in a long log. The *session* around it is the signature. Group the user's events by `(account_id, session_id)` and describe each session by what it started with: how much history the account had, how old its device was, whether its network was new, and what it did.

"New device" needs care. `is_new_device_id` is attached to the *first event that carries the device*, which in an MFA session is the `mfa_challenge_success`, not the `login_success`. Take the session's first event, not its login.

In [ ]:
# how long had this account known each device by each event?
dev_first = u.groupby(["account_id", "device_id"], observed=True).ts.min().rename("dev_first").reset_index()
u = u.merge(dev_first, on=["account_id", "device_id"], how="left")
u["dev_age_days"] = (u.ts - u.dev_first).dt.total_seconds() / 86400
u = u.sort_values(["ts", "event_id"])
g = u.groupby(["account_id", "session_id"], observed=True)
S = g.agg(start=("ts", "first"), first_event_id=("event_id", "first"), n_prior=("n_prior", "first"), dev_age_days=("dev_age_days", "first"), new_asn=("is_new_asn", "first"),
          has_login=("event_type", lambda s: (s == "login_success").any()), has_address=("event_type", lambda s: (s == "address_added").any())).reset_index()
orders = u[(u.event_type == "order_placed") & (u.d_ship_to_new_address == True)]
S = S.merge(orders.groupby(["account_id", "session_id"], observed=True).amount.max().rename("amount").reset_index(), on=["account_id", "session_id"])
print(len(S), "sessions that ordered to a new address,", S.account_id.nunique(), "accounts")

## 5.2 The rule, and the first mistake

Candidate session: a login, an `address_added`, an order to the new address, on an account with at least `MIN_HISTORY` earlier events, **from a device the account has only just started using**.

*Just started* is the interesting clause. Try "brand new at this session" first (`dev_age_days == 0`) and count how many of the phishing kit's victims it reaches. Then ask what the kit does in its *first* session on a new cookie, and widen the clause to `YOUNG_DAYS`. Mind the log's own start date: a device "first seen" in the log's first two weeks may be years old.

In [ ]:
day0 = df.ts.min().normalize()
# TODO: the candidate mask. History, a login and an address in the session, a young device,
# and a session that starts after the log's first YOUNG_DAYS days (device age is left-censored before that)
C = S.iloc[0:0].copy()
print(len(C), "candidate sessions on", C.account_id.nunique(), "accounts;", (~C.account_id.drop_duplicates().isin(prev.account_id)).sum(), "not already flagged")

## 5.3 Rank the queue

You will have more candidates than an analyst wants to open, and several honest cohorts qualify (gift buyers, new-phone upgraders, privacy browsers whose cookies are cleared, relocators). Rank by how unlike *this account's own* orders the order is: the amount against the median of its **earlier** orders (never include the order being scored in its own baseline), plus a small bonus when the network is new.

A phishing kit orders at about 1.6x the victim's usual amount, a low-and-slow operator at 2.5x. A gift buyer's December order is a normal order that happens to ship elsewhere.

In [ ]:
pop_median = u[u.event_type == "order_placed"].amount.median()
# TODO: baseline = median of this account's EARLIER orders (fall back to pop_median); score = log1p(ratio) + 0.5 * new_asn
C["score"] = 0.0
queue = C.sort_values("score", ascending=False).drop_duplicates("account_id")
queue = queue[~queue.account_id.isin(prev.account_id)].head(100)
queue[["account_id", "score", "n_prior", "dev_age_days", "new_asn"]].head(10)

## 5.4 The trade: MIN_HISTORY at 10, 5 and 3

Rerun 5.2-5.3 with `MIN_HISTORY` set to 10, then 5, then 3, and write down for each: queue size, takeovers found, **how many of the five phishing-kit victims Round 4 still missed** the queue reaches, and which honest cohorts it adds. The accounts the kit picks are thin in the first weeks; the clause that was free in Round 1 now costs recall.

In [ ]:
# TODO: wrap 5.2-5.3 in a function of MIN_HISTORY and print one line per setting
# queue size | takeovers | phishing-kit victims reached | honest accounts added

## 5.5 Submit (cumulative) and score the delta

In [ ]:
s = queue.score.to_numpy()
new = pd.DataFrame({"account_id": queue.account_id.to_numpy(), "score": 0.5 * s / (1 + s), "first_event_id": queue.first_event_id.to_numpy()})   # squashed below the earlier rounds' scores
out = pd.concat([prev[["account_id", "score", "first_event_id"]], new]).drop_duplicates("account_id")
save_suspects(out[["account_id"]], "../suspects_r5.csv", scores=out.score, first_event=out.first_event_id)
score_now("../suspects_r5.csv", "--prev", "../suspects_r4.csv")

## 5.6 Before the reveal

- Read the cohort table. A cohort can be inside its 2% budget on this round's delta and still go over because an earlier round had already spent the allowance. Which cohort, and whose fault is that?
- Which victims are still missing? What would the next attacker change? (They read this notebook too.)
- The rule uses *no* anomaly score. What does a hand-built sequence rule have that a model does not, and what does it cost to maintain?

Then `06_reveal.ipynb`, or `python tools/checkpoint.py --reveal`. Fell behind? `python tools/checkpoint.py --round 5 --catch-up`.